In [0]:
from pyspark.sql.types import StructType, StructField, IntegerType, StringType

# Users table data
users_data = [
    (1, "2019-01-01", "Lenovo"),
    (2, "2019-02-09", "Samsung"),
    (3, "2019-01-19", "LG"),
    (4, "2019-05-21", "HP")
]

users_schema = StructType([
    StructField("user_id", IntegerType(), True),
    StructField("join_date", StringType(), True),
    StructField("favorite_brand", StringType(), True)
])

users_df = spark.createDataFrame(users_data, schema=users_schema)
users_df.write.mode("overwrite").saveAsTable("practice_users")

# Orders table data
orders_data = [
    (1, "2019-08-01", 4, 1, 2),
    (2, "2019-08-02", 2, 1, 3),
    (3, "2019-08-03", 3, 2, 3),
    (4, "2019-08-04", 1, 4, 2),
    (5, "2019-08-04", 1, 3, 4),
    (6, "2019-08-05", 2, 2, 4)
]




orders_schema = StructType([
    StructField("order_id", IntegerType(), True),
    StructField("order_date", StringType(), True),
    StructField("item_id", IntegerType(), True),
    StructField("buyer_id", IntegerType(), True),
    StructField("seller_id", IntegerType(), True)
])



orders_df = spark.createDataFrame(orders_data, schema=orders_schema)
orders_df.write.mode("overwrite").saveAsTable("practice_orders")

items_data = [
    (1, "Samsung"),
    (2, "Lenovo"),
    (3, "LG"),
    (4, "HP")
]

items_schema = StructType([
    StructField("item_id", IntegerType(), True),
    StructField("item_brand", StringType(), True)
])

items_df = spark.createDataFrame(items_data, schema=items_schema)
items_df.write.mode("overwrite").saveAsTable("practice_items")
                                              
# Create SQL views for practice
spark.sql("CREATE OR REPLACE VIEW users AS SELECT * FROM practice_users")
spark.sql("CREATE OR REPLACE VIEW orders AS SELECT * FROM practice_orders")
spark.sql("CREATE OR REPLACE VIEW items AS SELECT * FROM practice_items")


spark.sql("""
    select * from users
""").display()

spark.sql("""
    select * from orders
""").display()

spark.sql("""
    select * from items
""").display()

In [0]:
spark.sql("""
    WITH ranked_orders as (
        select *, 
        RANK() OVER(partition by seller_id order by order_date) as rk
        from orders
    )
    select u.user_id, u.favorite_brand, I.item_brand,
    CASE WHEN u.favorite_brand = I.item_brand THEN 1 ELSE 0 END as is_fav from 
    users u left join ranked_orders r on u.user_id = r.seller_id and rk = 2
    LEFT JOIN items I on r.item_id = I.item_id
""").display()

In [0]:
import pyspark.sql.functions as F
from pyspark.sql.window import Window
ranked_orders = orders_df.withColumn(
    "rk", F.rank().over(Window.partitionBy("seller_id").orderBy(F.col('order_date')))
)

joined_df = users_df.alias("u").join(ranked_orders.alias("r"), \
     (F.col("u.user_id") == F.col("r.seller_id")) & (F.col("r.rk") == 2), "left").\
     join(items_df.alias("I"), F.col("r.item_id") == F.col("I.item_id"), "left").\
     select("u.user_id", "u.favorite_brand", "I.item_brand", \
            F.when(F.col("u.favorite_brand") == F.col("I.item_brand"), 1).otherwise(0).alias("is_fav"))


In [0]:
joined_df.show()